# TabFM vs GBM — 신용 대손 검증 (재현 노트북)

구글 **TabFM**(제로샷 테이블 파운데이션 모델)이 신용 대손 예측에서 잘 튜닝한 GBM을 이길까요. 공개 데이터(UCI 대만 신용카드 대손)로 겨뤄본 검증의 재현 코드입니다.

- 전체 글: **[han-co.com/ko/blog/tabfm-credit](https://han-co.com/ko/blog/tabfm-credit)**
- TabFM 원문: [구글 리서치 블로그](https://research.google/blog/introducing-tabfm-a-zero-shot-foundation-model-for-tabular-data/) · [GitHub](https://github.com/google-research/tabfm)

> **주의**
> - 이 노트북은 **모델(TabFM 가중치)을 포함하지 않습니다.** 실행하면 Hugging Face(`google/tabfm-1.0.0-pytorch`)에서 자동으로 받아옵니다(VRAM 약 6.5GB).
> - TabFM arm은 **CUDA GPU가 필요**합니다(CPU는 10배 이상 느림). 아래는 8GB GPU 기준 설정이며, 큰 GPU면 컨텍스트를 키우세요.
> - GBM arm만 보려면 TabFM 셀은 건너뛰어도 됩니다.

핵심 3개 arm(**날것 GBM · 튜닝 GBM · TabFM 제로샷**)을 보여줍니다. CatBoost·XGBoost·TabFM 앙상블까지 포함한 전체 6개 arm은 저장소의 `src/`(config 기반)에서 돌립니다.


## 실행 방법

Python 3.12 기준. 가상환경을 만들고 설치합니다.

```bash
python -m venv .venv
# Windows: .\.venv\Scripts\Activate.ps1   /   mac·linux: source .venv/bin/activate
pip install -r requirements.txt
# TabFM (CUDA torch 먼저, 그다음 소스 설치)
pip install torch --index-url https://download.pytorch.org/whl/cu124
pip install "tabfm[pytorch] @ git+https://github.com/google-research/tabfm.git"
```

작은 팁: TabFM은 GPU 메모리를 많이 써서, 이 노트북은 **데이터를 먼저 로드한 뒤** TabFM 셀에서 `torch`/`tabfm`을 import합니다.


In [ ]:
# 최초 1회만 (주석 해제). 위 '실행 방법' 참고. TabFM arm은 GPU + CUDA 필요.
# %pip install -q numpy pandas scikit-learn lightgbm optuna ucimlrepo
# %pip install -q torch --index-url https://download.pytorch.org/whl/cu124
# %pip install -q "tabfm[pytorch] @ git+https://github.com/google-research/tabfm.git"


## 1. 데이터 — UCI 대만 신용카드 대손

3만 명, 23개 피처, 다음 달 부도 여부(이진). 부도율 약 22%. `ucimlrepo`로 자동으로 받습니다. 컬럼은 위치 기준으로 표준 이름을 붙여, 로더가 달라도 뒤 코드가 항상 같은 컬럼을 찾게 합니다.


In [ ]:
import numpy as np, pandas as pd

CANONICAL = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
             "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
             *[f"BILL_AMT{i}" for i in range(1, 7)],
             *[f"PAY_AMT{i}"  for i in range(1, 7)]]
CATEG = ["SEX", "EDUCATION", "MARRIAGE"]

def load_uci():
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=350)
    X = ds.data.features.copy()
    X.columns = CANONICAL                                   # 위치 기준 표준화
    y = pd.to_numeric(ds.data.targets.iloc[:, 0]).astype(int).to_numpy()
    return X, y

X, y = load_uci()
print(X.shape, "| 부도율 =", round(float(y.mean()), 4))


## 2. 피처 엔지니어링 (튜닝 GBM용)

신용 애널리스트가 만들 법한 도메인 피처입니다. 이용률, 납부 커버리지, 연체 프로필·동태(연속 연체, 최근 대 과거 상태 변화 등). **이게 바로 TabFM이 상대하는 것**입니다. 제로샷 모델이 이 손수 만든 피처 없이 어디까지 따라오나를 봅니다. (TabFM에는 원본 23개 피처만 줍니다.)


In [ ]:
def engineer_credit_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy(); eps = 1.0
    limit = X["LIMIT_BAL"].astype(float)
    BILL = [f"BILL_AMT{i}" for i in range(1, 7)]
    AMT  = [f"PAY_AMT{i}"  for i in range(1, 7)]
    PAY  = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

    # 이용률: 한도 대비 청구액
    for i, c in enumerate(BILL, 1):
        X[f"UTIL_{i}"] = X[c].astype(float) / (limit + eps)
    util = [f"UTIL_{i}" for i in range(1, 7)]
    X["UTIL_MEAN"], X["UTIL_MAX"], X["UTIL_STD"] = X[util].mean(1), X[util].max(1), X[util].std(1)
    X["LAST_UTIL"] = X["UTIL_1"]

    # 납부 커버리지: 청구 대비 납부
    for i in range(1, 6):
        X[f"PAYRATIO_{i}"] = X[f"PAY_AMT{i}"].astype(float) / (X[f"BILL_AMT{i+1}"].astype(float).abs() + eps)
    pr = [f"PAYRATIO_{i}" for i in range(1, 6)]
    X["PAYRATIO_MEAN"], X["PAYRATIO_MIN"] = X[pr].mean(1), X[pr].min(1)

    # 연체 프로필 (연체상태 컬럼)
    pay = X[PAY].astype(float)
    X["PAY_MAX"], X["PAY_MEAN"], X["PAY_SUM"] = pay.max(1), pay.mean(1), pay.sum(1)
    X["N_DELAYED"] = (pay > 0).sum(1)
    X["N_SEVERE_DELAY"] = (pay >= 2).sum(1)
    X["PAY_TREND"] = pay["PAY_0"] - pay["PAY_6"]

    # 청구·납부 규모/추세
    bill, amt = X[BILL].astype(float), X[AMT].astype(float)
    X["BILL_MEAN"], X["BILL_STD"] = bill.mean(1), bill.std(1)
    X["BILL_TREND"] = bill["BILL_AMT1"] - bill["BILL_AMT6"]
    X["AMT_MEAN"], X["AMT_SUM"], X["AMT_STD"] = amt.mean(1), amt.sum(1), amt.std(1)
    X["ZERO_PAYMENTS"] = (amt == 0).sum(1)

    # 연체 동태: 연속 연체 streak, 마지막 연체 이후 개월, 최근3 vs 과거3
    p = X[PAY].to_numpy(float); d = (p > 0).astype(int)
    streak = np.zeros(len(X)); cur = np.zeros(len(X))
    for j in range(d.shape[1]):
        cur = (cur + d[:, j]) * d[:, j]
        streak = np.maximum(streak, cur)
    X["DELINQ_STREAK_MAX"] = streak
    X["MONTHS_SINCE_DELINQ"] = np.where(d.any(1), d.argmax(1), 6)
    X["PAY_STATUS_RECENT3"] = pay[["PAY_0", "PAY_2", "PAY_3"]].mean(1)
    X["PAY_STATUS_OLD3"]    = pay[["PAY_4", "PAY_5", "PAY_6"]].mean(1)
    X["PAY_STATUS_DELTA"]   = X["PAY_STATUS_RECENT3"] - X["PAY_STATUS_OLD3"]
    X["WORSENING"] = (pay["PAY_0"] > pay["PAY_6"]).astype(int)

    # 총 상환율 / 한도 상대
    X["TOTAL_PAID_TO_BILLED"] = amt.sum(1) / (bill.abs().sum(1) + eps)
    X["LIMIT_PER_AGE"] = limit / (X["AGE"].astype(float) + eps)

    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    return X.fillna(0.0)

print("엔지니어링 후 피처 수:", engineer_credit_features(X).shape[1])


## 3. 평가지표 — 판별 + 캘리브레이션

신용은 순위(판별)뿐 아니라 확률(캘리브레이션)도 중요합니다. 판별은 **ROC-AUC · PR-AUC · KS**, 캘리브레이션은 **Brier · LogLoss · ECE**로 봅니다. PR-AUC는 드문 부도를 얼마나 잡는지, ECE는 예측 확률이 실제 부도율과 얼마나 맞는지를 봅니다.


In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             brier_score_loss, log_loss, roc_curve)

def ks_statistic(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.max(tpr - fpr))

def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(y_prob, edges[1:-1]), 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b; cnt = int(m.sum())
        if cnt:
            ece += (cnt / len(y_true)) * abs(y_true[m].mean() - y_prob[m].mean())
    return float(ece)

def compute_metrics(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.clip(np.asarray(y_prob, float), 1e-7, 1 - 1e-7)
    auc = roc_auc_score(y_true, y_prob)
    return {"roc_auc": float(auc), "gini": float(2 * auc - 1),
            "pr_auc": float(average_precision_score(y_true, y_prob)),
            "ks": ks_statistic(y_true, y_prob),
            "brier": float(brier_score_loss(y_true, y_prob)),
            "log_loss": float(log_loss(y_true, y_prob)),
            "ece": expected_calibration_error(y_true, y_prob)}


## 4. 모델 — 세 arm

- **날것 GBM**: LightGBM 기본값, 피처·튜닝 없음 (노력의 바닥선)
- **튜닝 GBM**: 엔지니어링 피처 + Optuna 튜닝 + early stopping (강한 기준선)
- **TabFM 제로샷**: 아웃오브박스, 튜닝 0. Hugging Face에서 받아 한 번의 forward pass로 예측

**공정 비교 원칙**: 확률이 실제 부도율과 맞도록 `class_weight`를 주지 않고 자연 비율로 학습합니다(캘리브레이션 공정 비교). 판별 지표는 이 선택에 거의 불변입니다.

**8GB GPU 설정(TabFM)**: 가중치만 6.5GB라 컨텍스트를 1,000행으로 캡하고, estimator 32개로 서로 다른 1,000행씩 보게 해 학습셋을 사실상 다 커버합니다(peak VRAM은 그대로). 테스트 예측은 500행씩 나눠 합니다. **16GB 이상이면** `CONTEXT_MAX`를 키우고 앙상블 프리셋을 쓰세요.


In [ ]:
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier, early_stopping
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def gbm_raw_proba(Xtr, ytr, Xte):
    for c in CATEG:
        Xtr[c] = Xtr[c].astype("category"); Xte[c] = Xte[c].astype("category")
    m = LGBMClassifier(n_estimators=300, learning_rate=0.05, objective="binary",
                       n_jobs=-1, verbosity=-1, random_state=42)
    m.fit(Xtr, ytr)
    return m.predict_proba(Xte)[:, 1]

OPTUNA_TRIALS = 40   # 블로그 실험은 80. 노트북에선 시간 고려해 40 (원하면 올리세요).

def gbm_full_proba(Xtr, ytr, Xte, seed=42):
    Xtr, Xte = engineer_credit_features(Xtr), engineer_credit_features(Xte)
    for c in CATEG:
        Xtr[c] = Xtr[c].astype("category"); Xte[c] = Xte[c].astype("category")
    inner = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(Xtr, ytr))

    def objective(t):
        p = dict(learning_rate=t.suggest_float("learning_rate", 0.01, 0.2, log=True),
                 num_leaves=t.suggest_int("num_leaves", 15, 255),
                 max_depth=t.suggest_int("max_depth", 3, 12),
                 min_child_samples=t.suggest_int("min_child_samples", 5, 120),
                 subsample=t.suggest_float("subsample", 0.6, 1.0),
                 colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
                 reg_alpha=t.suggest_float("reg_alpha", 1e-8, 10, log=True),
                 reg_lambda=t.suggest_float("reg_lambda", 1e-8, 10, log=True))
        sc, it = [], []
        for tr, va in inner:
            m = LGBMClassifier(n_estimators=2500, objective="binary", n_jobs=-1,
                               verbosity=-1, random_state=seed, subsample_freq=1, **p)
            m.fit(Xtr.iloc[tr], ytr[tr], eval_set=[(Xtr.iloc[va], ytr[va])],
                  eval_metric="auc", callbacks=[early_stopping(80, verbose=False)])
            sc.append(roc_auc_score(ytr[va], m.predict_proba(Xtr.iloc[va])[:, 1]))
            it.append(m.best_iteration_ or 2500)
        t.set_user_attr("n_est", float(np.mean(it)))
        return float(np.mean(sc))

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=OPTUNA_TRIALS)
    n_est = max(50, min(2500, int(round(study.best_trial.user_attrs["n_est"] * 1.05))))
    m = LGBMClassifier(n_estimators=n_est, objective="binary", n_jobs=-1, verbosity=-1,
                       random_state=seed, subsample_freq=1, **study.best_params)
    m.fit(Xtr, ytr)
    return m.predict_proba(Xte)[:, 1]


In [ ]:
# --- TabFM 제로샷 (CUDA GPU 필요; 여기서 torch/tabfm를 import) ---
CONTEXT_MAX  = 1000     # 8GB 기준. 16GB+면 키우세요 (예: 12000).
N_ESTIMATORS = 32       # 논문 기본값. 각 estimator가 다른 1,000행을 봐 학습셋을 커버.
PREDICT_BATCH = 500

def as_tabfm_frame(X):
    X = X.copy()
    for c in X.columns:
        X[c] = X[c].astype(str) if c in CATEG else pd.to_numeric(X[c], errors="coerce").astype(float)
    return X

_TABFM = None
def _load_tabfm():
    global _TABFM
    if _TABFM is None:
        from tabfm import tabfm_v1_0_0_pytorch as T
        _TABFM = T.load(device="cuda")      # HF에서 가중치 다운로드 (~6.5GB VRAM)
    return _TABFM

def tabfm_zeroshot_proba(Xtr, ytr, Xte):
    from tabfm import TabFMClassifier
    clf = TabFMClassifier(model=_load_tabfm(), n_estimators=N_ESTIMATORS,
                          max_num_rows=CONTEXT_MAX, random_state=42, verbose=False)
    clf.fit(as_tabfm_frame(Xtr), ytr)
    Xte = as_tabfm_frame(Xte)
    parts = [clf.predict_proba(Xte.iloc[i:i + PREDICT_BATCH])[:, 1]   # 메모리 안전 배치 예측
             for i in range(0, len(Xte), PREDICT_BATCH)]
    return np.concatenate(parts)


## 5. 교차검증 (층화 5-fold)

같은 분할로 세 arm을 돌립니다. TabFM은 GPU가 없으면 이 셀에서 오래 걸리거나 실패하니, GBM만 보려면 `ARMS`에서 `tabfm_zeroshot`을 빼세요.


In [ ]:
import time

ARMS = {"gbm_raw": gbm_raw_proba, "gbm_full": gbm_full_proba,
        "tabfm_zeroshot": tabfm_zeroshot_proba}
# GPU가 없으면: ARMS.pop("tabfm_zeroshot")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, fn in ARMS.items():
    for k, (tr, te) in enumerate(skf.split(X, y)):
        t0 = time.time()
        p = fn(X.iloc[tr].copy(), y[tr], X.iloc[te].copy())
        m = compute_metrics(y[te], p)
        m.update(arm=name, fold=k, sec=round(time.time() - t0, 1))
        rows.append(m)
        print(f"{name:16s} fold{k}: AUC={m['roc_auc']:.4f}  KS={m['ks']:.3f}  ECE={m['ece']:.3f}")
res = pd.DataFrame(rows)


## 6. 결과


In [ ]:
order = [a for a in ["gbm_full", "tabfm_zeroshot", "gbm_raw"] if a in res["arm"].unique()]
summary = (res.groupby("arm")[["roc_auc", "pr_auc", "ks", "brier", "ece", "sec"]]
             .mean().round(4).reindex(order))
summary


## 정리

- 잘 튜닝한 GBM이 TabFM 제로샷을 근소하게 앞섭니다 (블로그 실험 기준 AUC 0.789 대 0.785, 폴드 노이즈 안).
- 무노력끼리(날것 GBM 대 TabFM 제로샷)면 TabFM이 앞섭니다.
- 캘리브레이션은 대등합니다 (`class_weight` 없이 학습했을 때).

**전체 6개 arm**(CatBoost·XGBoost·TabFM 앙상블 포함)과 정확한 재현 설정은 저장소의 `src/`와 `config.yaml`에 있습니다. **TabFM 앙상블 프리셋**(cross+SVD 피처 + NNLS 가중 + Platt 캘리브레이션)은 큰 컨텍스트가 필요해 **16GB 이상 GPU**를 권합니다.

전체 글: **[han-co.com/ko/blog/tabfm-credit](https://han-co.com/ko/blog/tabfm-credit)**
